# OCR Extraction Pipeline — Setup & Test

End-to-end extraction pipeline for university grant administration documents using a Vision Language Model (VLM).

**What this notebook does:**
1. Loads a VLM (Qwen3-VL-32B or Qwen2.5-VL-72B) onto GPU
2. Renders PDF pages as images and sends each to the VLM for structured extraction
3. Extracts: narratives (verbatim), tables, stakeholders, addresses, signatures, annotations
4. Assembles per-page results into a document-level JSONL matching the grant admin schema
5. Batch processes all PDFs in `/ocr/`
6. Compares VLM output against Gemini reference JSONs with plots

**Before starting:** Upload your sample PDFs to `/ocr/` and Gemini reference JSONs to `/ocr/gemini/` using Jupyter's file upload button.


## 1. Check GPU and shared models

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path
import os

os.environ["HF_HOME"] = "/models/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/models/.cache/huggingface"
os.environ["HF_HUB_OFFLINE"] = "1"

models_dir = Path("/models/.cache/huggingface")
if models_dir.exists():
    model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
    print(f"Shared models PVC mounted. {len(model_dirs)} model(s):")
    for m in sorted(model_dirs):
        print(f"  {m}")
    has_vlm = any("Qwen3-VL" in d or "Qwen2.5-VL" in d for d in model_dirs)
    if has_vlm:
        print("\nQwen VL model found.")
    else:
        print("\nWARNING: No Qwen VL model found (looked for Qwen3-VL and Qwen2.5-VL)!")
        print("Run: python /models/provision_shared_models.py download QuantTrio/Qwen3-VL-32B-Instruct-AWQ")
else:
    print("WARNING: /models/.cache/huggingface not found.")
    print("Is the shared-models data volume attached?")


## 2. Load the model

Change `MODEL_NAME` below to switch models. The `Auto` class handles both Qwen2.5-VL and Qwen3-VL architectures automatically.

**GPU memory requirements (bf16):**
- Qwen3-VL-32B: ~64 GB (fits on 96 GB GPU)
- Qwen2.5-VL-72B: ~145 GB (needs AWQ quantized version for single GPU)
- Qwen3-VL-8B: ~16 GB


In [ ]:
import time
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

# ── Choose a model (uncomment one) ────────────────────────────────────
# MODEL_NAME = "Qwen/Qwen3-VL-32B-Instruct"          # ~64 GB bf16 — full precision
MODEL_NAME = "QuantTrio/Qwen3-VL-32B-Instruct-AWQ"  # ~20 GB INT4 — default, same 32B quality
# MODEL_NAME = "Qwen/Qwen2.5-VL-72B-Instruct-AWQ"  # ~40 GB INT4 — best quality, fits 96 GB
# MODEL_NAME = "Qwen/Qwen2.5-VL-72B-Instruct"      # ~137 GB bf16 — needs multi-GPU or quantized
# MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"          # ~16 GB bf16 — fast, lower quality
# MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"        # ~15 GB bf16 — fast, lower quality

# ── Quantization config ──────────────────────────────────────────────
# Pre-quantized AWQ models load automatically — just uncomment above.
# For bf16 models on smaller GPUs, set USE_BNB_4BIT = True to quantize
# on-the-fly (slower to load, same inference speed).
USE_BNB_4BIT = False
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
) if USE_BNB_4BIT else None

print(f"Loading {MODEL_NAME}...")
if bnb_config:
    print("  (with BitsAndBytes 4-bit quantization)")
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
    quantization_config=bnb_config,
)

elapsed = time.time() - t0
print(f"Model loaded in {elapsed:.1f}s")
print(f"Device: {model.device}")

# GPU memory report
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        used = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        free = total - reserved
        print(f"GPU {i}: {used:.1f} GB allocated / {reserved:.1f} GB reserved / {total:.1f} GB total ({free:.1f} GB free)")


### Install VLM utilities

`qwen-vl-utils` provides `process_vision_info()` which preprocesses images/videos for Qwen VL models. Only needs to run once per environment.


In [ ]:
!pip install -q uv
!uv pip install --system qwen-vl-utils autoawq


### Define helper functions

- `run_vlm(messages)` — core inference: tokenizes input, runs model, decodes output
- `extract_page(image, prompt)` — encodes a page image as base64 PNG and sends it with the extraction prompt to the VLM


In [ ]:
from qwen_vl_utils import process_vision_info

def run_vlm(messages, max_tokens=4096):
    """Run inference on the loaded model."""
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

def extract_links(page):
    """Extract hyperlinks from a PDF page. Returns list of {text, url}."""
    links = []
    for link in page.get_links():
        uri = link.get("uri", "")
        if not uri:
            continue
        # Try to get the link text from the rect area
        rect = link.get("from", None)
        text = page.get_text("text", clip=rect).strip() if rect else ""
        links.append({"text": text, "url": uri})
    return links

def extract_page(image, prompt, max_tokens=4096, filename=None, links=None):
    """Send a rendered page image to the VLM for structured extraction."""
    import base64, io
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()

    # Build context prefix
    context_parts = []
    if filename:
        context_parts.append(f"Source filename: {filename}")
    if links:
        link_lines = "\n".join(f"  {l['text']} -> {l['url']}" for l in links)
        context_parts.append(f"Hyperlinks found in this page:\n{link_lines}")
    prefix = "\n\n".join(context_parts)
    text = f"{prefix}\n\n{prompt}" if prefix else prompt

    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"data:image/png;base64,{b64}"},
        {"type": "text", "text": text},
    ]}]
    return run_vlm(messages, max_tokens)

print("Helper functions ready.")


### Extraction prompt, filename parser, and assembly function

This cell defines three things:

1. **`EXTRACTION_PROMPT`** — the per-page prompt sent to the VLM. Based on the grant admin spec, it asks for: narratives (verbatim with `[cite: N]`), tables (3 classification types), stakeholders, addresses, signatures, annotations, watermarks, confidence, and document details.

2. **`parse_filename()`** — parses structured filenames like `A_RSP Public_AWD-001064_3718296725__AWD00000002__A_RSP_Award_Notice_of_Award.pdf` into metadata fields (Drawer, AwardID, Field2-5, DocumentType). Double underscores (`__`) indicate empty fields.

3. **`assemble_document_jsonl()`** — merges per-page VLM results into the final document-level JSON matching the spec. Maps snake_case VLM output to PascalCase schema, aggregates across pages.


In [ ]:
# ── Extraction prompt (per-page) ──────────────────────────────────────
# Adapted from the full document-level prompt for page-by-page processing.
# The assembly function merges per-page results into the final JSONL.

EXTRACTION_PROMPT = """Role & Objective: You are an expert data extraction assistant specializing in university grant administration documents. Data quality is paramount: do not abbreviate, shorten, or use external assumptions to fill in missing values.

Task: Extract all data from this single document page into the JSON structure below. Do not analyze data.

{
  "confidence_percentage": <float 0-100>,
  "confidence_narrative": "<brief note on extraction quality and comprehensiveness>",
  "has_annotation": <boolean>,
  "has_watermark": <boolean>,
  "signature_lines": {
    "has_signature_line": <boolean>,
    "has_valid_signature": <boolean>
  },
  "document_tags": ["<high-level grant admin tags, e.g. IRB, IACUC, Biosafety>"],
  "one_sentence_summary": "<one sentence summary>",
  "document_details": {
    "application_id": "", "application_type": "", "title": "",
    "requested_amount": null, "completed_date": "", "sub_document_type": ""
  },
  "stakeholders": [
    {
      "context_snippet": "<3-5 words near the stakeholder info>",
      "stakeholder_role": "<Principal Investigator | Co-Investigator | Collaborator | Key Personnel | Grants Administrative Contact | Sponsor Contact | Authorized Organizational Representative | Unknown>",
      "full_name": "", "first_name": "", "last_name": "",
      "email": "", "phone": "", "institution": "", "department": "",
      "position_title": "", "highest_education": "",
      "raw_stakeholder_text": "<verbatim text block containing stakeholder info>"
    }
  ],
  "addresses": [
    {
      "context_snippet": "<3-5 words near the address>",
      "addressee": "", "care_of": null,
      "address_line1": "", "address_line2": "",
      "city": "", "state_province": "", "postal_code": "", "country": "",
      "stakeholder_type": "<Funding Agency | Grantee Institution | Subrecipient | Principal Investigator | Grants Administrative Contact | Unknown>",
      "raw_address_text": "<verbatim text block containing the full address>"
    }
  ],
  "tables": [
    {
      "table_classification": "<Literal_Grid | Key_Value_Form | Standard_Table>",
      "table_data": "<see classification rules below>"
    }
  ],
  "narrative_responses": [
    {
      "prompt_or_header": "<exact question, section header, or 'General Body Text'>",
      "verbatim_text": "<complete, unsummarized text with [cite: N] markers>"
    }
  ],
  "continuation": {
    "continues_from_previous": "<true if content clearly continues from a prior page (e.g. mid-sentence, table header on previous page)>",
    "continues_to_next": "<true if content is clearly cut off and continues on the next page>",
    "continuation_type": "<null | text | table | list | narrative>"
  },
  "other_metadata": {}
}

PROCESSING RULES:
- STAKEHOLDER EXTRACTION (CRITICAL): Every grant document has AT LEAST two stakeholders: a funding agency/sponsor AND a recipient institution. ALWAYS extract both. The funding/granting agency (e.g. NSF, NIH, NHPRC, DOE) must be extracted as a stakeholder with role "Sponsor Contact" even if only mentioned in a header, award number prefix, or letterhead. The recipient institution must be extracted as role "Authorized Organizational Representative" or "Unknown". Also extract all individuals mentioned as investigators, collaborators, points of contact, or key personnel.
- NARRATIVE EXTRACTION (CRITICAL FOR RAG): Extract ALL body text, paragraphs, memos, and application answers VERBATIM to ensure 100% document coverage. If text is part of a Q&A form, include the question in prompt_or_header. For unstructured letter/memo body, use "General Body Text". Do NOT summarize, truncate, or condense.
- CITATIONS: Add [cite: N] numbered tags after each distinct statement in narrative text, incrementing N from 1.
- TABLE CLASSIFICATION:
  * Literal_Grid: irregular tables without clear headers — table_data is a 2D array of strings (list of rows).
  * Key_Value_Form: label-value pairs (e.g. form cover sheets) — table_data is a single JSON object {label: value}.
  * Standard_Table: clear column headers — table_data is an array of objects with column headers as keys.
- SIGNATURES: Do NOT read handwriting. Only note if a signature LINE exists and if a signature is DETECTED.
- STAKEHOLDER ROLES: Use ONLY the allowed stakeholder_role values listed above. If context does not make the role explicitly clear, use "Unknown". Capture raw_stakeholder_text verbatim.
- HYPERLINKS: If hyperlinks are provided in the context above, include them in the relevant narrative text or other_metadata. Preserve the exact URL.
- ADDRESSES: Use ONLY the allowed stakeholder_type values listed above. If unclear, use "Unknown". Place "Care Of"/"Attention" lines ONLY in care_of. Capture raw_address_text verbatim.
- Preserve ALL dollar amounts, dates, reference numbers exactly as they appear.
- CONTINUATION: If text, a table, or a list appears to be cut off at the top or bottom of the page, set the appropriate continuation flags. This helps downstream assembly merge cross-page content. If you cannot tell, set both to false.
- Missing fields: use null or "" as appropriate. Escape all strings.
- Output ONLY valid JSON. No markdown fences, no introductory text."""


# ── Filename metadata parser ─────────────────────────────────────────
def parse_filename(filename):
    """Parse structured filename into FileNameMetaData.

    Rules from spec:
    - Six data elements in strict order, OFTEN split by single underscore.
    - Two underscores (__) next to each other = that data element is "".
    - After AwardID: Field2, Field3, Field4, Field5, then DocumentType.
    """
    import re
    stem = Path(filename).stem
    ext = Path(filename).suffix.lstrip('.')

    awd_match = re.search(r'_AWD-', stem)
    if not awd_match:
        return {
            "Drawer": stem, "AwardID": "", "Field2": "", "Field3": "",
            "Field4": "", "Field5": "", "DocumentType": stem,
            "DocumentTypeShort": stem, "FileType": ext
        }

    drawer = stem[:awd_match.start()]
    rest = stem[awd_match.start() + 1:]  # drop leading _

    # Split on _ — empty strings from __ indicate empty fields
    parts = rest.split('_')
    award_id = parts[0] if parts else ""

    # After AwardID: next 4 positions are Field2-Field5, rest is DocumentType
    remaining = parts[1:]
    field2 = remaining[0] if len(remaining) > 0 else ""
    field3 = remaining[1] if len(remaining) > 1 else ""
    field4 = remaining[2] if len(remaining) > 2 else ""
    field5 = remaining[3] if len(remaining) > 3 else ""
    doc_type = '_'.join(remaining[4:]) if len(remaining) > 4 else ""

    # DocumentTypeShort: strip category prefix (e.g. "A_RSP_Award_" -> "Notice_of_Award")
    doc_type_short = doc_type
    for cat in ("_Award_", "_Budget_", "_Report_", "_Agreement_", "_Proposal_"):
        idx = doc_type.find(cat)
        if idx >= 0:
            doc_type_short = doc_type[idx + len(cat):]
            break

    return {
        "Drawer": drawer, "AwardID": award_id,
        "Field2": field2, "Field3": field3, "Field4": field4, "Field5": field5,
        "DocumentType": doc_type, "DocumentTypeShort": doc_type_short,
        "FileType": ext
    }


# ── Document-level assembly (produces JSONL-ready dict) ──────────────
def assemble_document_jsonl(filename, page_results, model_name, prompt_text=None):
    """Combine per-page VLM results into the final document-level JSON per spec.

    Pass 1 results are the source of truth — this function never modifies them.
    It adds: document-level metadata, cross-page linkage, confidence summary,
    and the prompt used for reproducibility."""
    from datetime import datetime, timezone

    file_meta = parse_filename(filename)

    all_tables, all_narratives = [], []
    all_stakeholders, all_addresses = [], []
    all_tags = set()
    summaries = []
    has_annotation = has_watermark = False
    sig_info = {"PageNumber": None, "HasSignatureLine": False, "HasValidSignature": False}
    confidence_scores, page_confidences = [], []
    doc_details = {}
    other_meta = {}

    for pr in page_results:
        pg = pr["page"]
        d = pr.get("extracted", {})

        # Tables — map per-page snake_case to PascalCase
        for t in d.get("tables", []):
            all_tables.append({
                "PageNumber": pg,
                "TableClassification": t.get("table_classification",
                                             t.get("classification", "Standard_Table")),
                "TableData": t.get("table_data", t.get("rows", []))
            })

        # Narratives
        for n in d.get("narrative_responses", []):
            all_narratives.append({
                "SectionOrPage": f"PAGE {pg}",
                "PromptOrHeader": n.get("prompt_or_header", ""),
                "VerbatimText": n.get("verbatim_text", "")
            })

        # Stakeholders — map to PascalCase with full fields
        for s in d.get("stakeholders", []):
            if not any(v for v in s.values() if v):
                continue
            all_stakeholders.append({
                "PageNumber": pg,
                "ContextSnippet": s.get("context_snippet", ""),
                "StakeholderRole": s.get("stakeholder_role", "Unknown"),
                "FullName": s.get("full_name", ""),
                "FirstName": s.get("first_name", ""),
                "LastName": s.get("last_name", ""),
                "Email": s.get("email", ""),
                "Phone": s.get("phone", ""),
                "Institution": s.get("institution", ""),
                "Department": s.get("department", ""),
                "PositionTitle": s.get("position_title", ""),
                "HighestEducation": s.get("highest_education", ""),
                "RawStakeholderText": s.get("raw_stakeholder_text", "")
            })

        # Addresses — map to PascalCase with full fields
        for a in d.get("addresses", []):
            if not any(v for v in a.values() if v):
                continue
            all_addresses.append({
                "PageNumber": pg,
                "ContextSnippet": a.get("context_snippet", ""),
                "Addressee": a.get("addressee", ""),
                "CareOf": a.get("care_of"),
                "AddressLine1": a.get("address_line1", ""),
                "AddressLine2": a.get("address_line2", ""),
                "City": a.get("city", ""),
                "StateProvince": a.get("state_province", ""),
                "PostalCode": a.get("postal_code", ""),
                "Country": a.get("country", ""),
                "StakeholderType": a.get("stakeholder_type", "Unknown"),
                "RawAddressText": a.get("raw_address_text", "")
            })

        all_tags.update(d.get("document_tags", []))
        if d.get("one_sentence_summary"):
            summaries.append(d["one_sentence_summary"])
        if d.get("has_annotation"): has_annotation = True
        if d.get("has_watermark"): has_watermark = True

        sig = d.get("signature_lines", {})
        if sig.get("has_signature_line"):
            sig_info = {"PageNumber": pg, "HasSignatureLine": True,
                        "HasValidSignature": sig.get("has_valid_signature", False)}

        if d.get("confidence_percentage") is not None:
            confidence_scores.append(d["confidence_percentage"])
        page_confidences.append({
            "PageNumber": pg,
            "ConfidencePercentage": d.get("confidence_percentage", 0),
            "ConfidenceNarrative": d.get("confidence_narrative", ""),
        })

        if not doc_details and d.get("document_details"):
            doc_details = d["document_details"]

        if d.get("other_metadata"):
            other_meta.update(d["other_metadata"])

    avg_conf = round(sum(confidence_scores) / len(confidence_scores), 1) if confidence_scores else 0.0
    now = datetime.now().astimezone().isoformat(timespec="seconds")

    # ── Cross-page linking (programmatic, no LLM) ─────────────
    # Detect pages whose continuation flags indicate content spans
    # page boundaries. This is metadata only — per-page results
    # are never modified.
    cross_page_links = []
    for i, pr in enumerate(page_results):
        d = pr.get("extracted", {})
        cont = d.get("continuation", {})
        if cont.get("continues_to_next"):
            next_pg = page_results[i + 1]["page"] if i + 1 < len(page_results) else None
            cross_page_links.append({
                "from_page": pr["page"],
                "to_page": next_pg,
                "type": cont.get("continuation_type", "unknown"),
            })

    # Build a concise top-level summary from per-page data
    low_conf = [pc for pc in page_confidences if pc["ConfidencePercentage"] < 80]
    failed = [pc for pc in page_confidences if not pc["ConfidenceNarrative"]
              or "Failed" in pc["ConfidenceNarrative"]]
    if low_conf or failed:
        problem_pages = sorted(set(
            pc["PageNumber"] for pc in (low_conf + failed)
        ))
        summary_narrative = (
            f"{len(page_results)} pages processed, avg confidence {avg_conf}%. "
            f"{len(low_conf)} page(s) below 80% confidence. "
            f"Problem pages: {problem_pages}."
        )
    else:
        summary_narrative = (
            f"{len(page_results)} pages processed, avg confidence {avg_conf}%. "
            f"All pages extracted successfully."
        )

    return {
        "Filename": filename,
        "PageCount": len(page_results),
        "ConfidencePercentage": avg_conf,
        "ConfidenceSummary": summary_narrative,
        "PageConfidences": page_confidences,
        "CrossPageLinks": cross_page_links,
        "ExtractionPrompt": prompt_text,
        "LLMModelAndVersion": model_name,
        "CurrentDateAndTime": now,
        "HasAnnotation": has_annotation,
        "HasWatermark": has_watermark,
        "SignatureLines": sig_info,
        "DocumentTags": sorted(all_tags),
        "OneSentenceNarrativeSummary": summaries,
        "FileNameMetaData": file_meta,
        "DocumentDetails": {
            "ApplicationID": doc_details.get("application_id", ""),
            "ApplicationType": doc_details.get("application_type", ""),
            "Title": doc_details.get("title", ""),
            "RequestedAmount": doc_details.get("requested_amount"),
            "CompletedDate": doc_details.get("completed_date", ""),
            "SubDocumentType": doc_details.get("sub_document_type", "")
        },
        "Stakeholders": all_stakeholders,
        "AddressesCollection": all_addresses,
        "TablesCollection": all_tables,
        "NarrativeResponses": all_narratives,
        "OtherMetadata": other_meta
    }


print("Extraction prompt, filename parser, and assembly function ready.")


### Alternative prompt: Library & archival materials

The default `EXTRACTION_PROMPT` above is tuned for grant administration documents.
For **library scans** — books, manuscripts, sheet music, newspapers, maps,
photographs, pamphlets, multilingual materials — use `LIBRARY_PROMPT` instead.

**To switch:** replace `EXTRACTION_PROMPT` with `LIBRARY_PROMPT` in the
extraction cells below. The JSON schema captures page type, full verbatim
transcription, musical notation, marginalia, library stamps/bookplates,
and physical condition — things grant docs don't have.

Also available as the `library` format in the extraction server and Streamlit app.

In [ ]:
# ── Library / archival prompt (per-page) ────────────────────────────
# For scanned books, manuscripts, sheet music, newspapers, maps,
# photographs, pamphlets, and other archival materials.
# Handles multilingual text, musical notation, marginalia, stamps.

LIBRARY_PROMPT = """Role & Objective: You are an expert transcription and cataloging assistant specializing in digitized library and archival materials. Accuracy is paramount: transcribe text exactly as printed — do not modernize spelling, correct errors, or expand abbreviations.

Task: Extract all information from this single scanned page into the JSON structure below.

{
  "page_type": "<book_page | title_page | table_of_contents | index | manuscript | sheet_music | newspaper | map | photograph | illustration | plate | form | correspondence | ephemera | blank | mixed>",
  "confidence_percentage": <float 0-100>,
  "one_sentence_summary": "<brief description of what this page contains>",

  "bibliographic": {
    "title": "<running title, chapter title, article headline, or piece title visible on this page>",
    "creator": "<author, composer, artist, editor, or cartographer if shown>",
    "date": "<any date visible: publication, manuscript, postmark, etc.>",
    "language": "<primary language of text — use ISO 639-1 code if clear, otherwise spell out>",
    "other_languages": ["<additional languages if multilingual>"],
    "publisher": "<publisher or printer if shown>",
    "place": "<place of publication or origin>"
  },

  "page_number": "<printed, stamped, or handwritten page/folio number>",
  "header": "<running header text if present>",
  "body_text": "<COMPLETE verbatim text in reading order — every word on the page>",

  "structure": {
    "headings": ["<chapter titles, section headings, article headlines>"],
    "footnotes": ["<footnote or endnote text, preserving reference marks>"],
    "lists": ["<any enumerated or bulleted lists>"],
    "poetry_verse": "<verse text preserving original line breaks, or null>"
  },

  "tables": [
    {
      "caption": "<table caption if any>",
      "table_data": "<2D array of cell values, or key-value object for forms>"
    }
  ],

  "visual_elements": {
    "illustrations": [
      {"description": "<what the image depicts>", "caption": "<printed caption>", "position": "<top|bottom|left|right|center|full_page>"}
    ],
    "musical_notation": {
      "present": false,
      "instrument_voice": "<e.g. Piano, Soprano, Violin I>",
      "key_signature": "<e.g. G major, C minor>",
      "time_signature": "<e.g. 4/4, 3/4, 6/8>",
      "tempo_markings": "<e.g. Allegro, Andante, ♩=120>",
      "dynamics": ["<pp, mf, crescendo, etc.>"],
      "lyrics": "<sung text under the notation, preserving syllable breaks>",
      "description": "<general description of what is notated>"
    },
    "maps": {"present": false, "region": "", "description": ""}
  },

  "marginalia": [
    {"text": "<transcription or description>", "location": "<top|bottom|left|right|interlinear>", "writing_instrument": "<pencil|pen|ink|stamp>"}
  ],

  "stamps_marks": [
    {"text": "<text on stamp/label>", "type": "<library_stamp | bookplate | accession_number | call_number | barcode | ownership_mark | censor_mark | other>"}
  ],

  "physical_condition": {
    "damage": "<describe any tears, stains, foxing, fading, water damage, or missing portions>",
    "legibility": "<fully_legible | mostly_legible | partially_legible | poor>",
    "scan_issues": "<skew, shadow from binding, cropped edges, bleed-through from verso>"
  },

  "other_metadata": {}
}

PROCESSING RULES:
- TRANSCRIPTION (CRITICAL): Transcribe ALL printed and typed text VERBATIM into body_text. Do not skip, summarize, or paraphrase. Preserve original spelling, punctuation, capitalization, and line breaks for poetry/verse.
- LANGUAGES: If the page contains multiple languages, set language to the primary one and list others in other_languages. Transcribe all languages as-is.
- SHEET MUSIC: Describe the musical content (instrument, key, tempo, dynamics, lyrics) but do not attempt to encode notes. If lyrics are present under the notation, transcribe them in musical_notation.lyrics with syllable hyphens preserved.
- HANDWRITING & MARGINALIA: Transcribe if legible. If not, describe what you see (e.g. "[illegible annotation in pencil, approximately 5 words, possibly a name]"]. Note the writing instrument and location.
- STAMPS & MARKS: Capture EVERY library stamp, bookplate, accession number, call number, and barcode — these are critical for provenance and cataloging.
- ILLUSTRATIONS: Describe what is depicted, note any printed caption, and indicate position on page.
- DAMAGE: Note anything that affects legibility — stains, tears, fading, tight binding shadow, bleed-through.
- Missing fields: use null or "" as appropriate.
- Output ONLY valid JSON. No markdown fences, no introductory text."""

print(f"Library prompt ready ({len(LIBRARY_PROMPT)} chars).")
print("To use: replace EXTRACTION_PROMPT with LIBRARY_PROMPT in the extraction cells below.")


## 3. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload button.

In [ ]:
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print("Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
DOC_PATH = Path("/ocr/" + f.name)

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Render pages

All pages are sent as images to the VLM (captures layout, tables, signatures, watermarks).

In [ ]:
import fitz
from PIL import Image

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_images = []
page_links = []
for i, page in enumerate(doc):
    mat = fitz.Matrix(1.0, 1.0)
    pix = page.get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    page_images.append(img)

    links = extract_links(page)
    page_links.append(links)

    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    label = "digital" if has_text else "scanned"
    link_info = f", {len(links)} link(s)" if links else ""
    print(f"Page {i+1}: {img.width}x{img.height} ({label}, {len(text)} chars{link_info})")

doc.close()
print(f"\n{len(page_images)} page(s) rendered. All will be sent as images to VLM.")


### Test extraction on a single page

Quick sanity check — renders one page and sends it to the VLM. Change `PAGE_IDX` to test different pages. Review the raw JSON output in the next cell before running the full batch.


In [ ]:
PAGE_IDX = 0  # Change this to test different pages
img = page_images[PAGE_IDX]

print(f"Page {PAGE_IDX+1}: {img.width}x{img.height}")
display(img.resize((img.width // 3, img.height // 3)))

t0 = time.time()
raw_result = extract_page(img, EXTRACTION_PROMPT, filename=DOC_PATH.name, links=page_links[PAGE_IDX])
elapsed = time.time() - t0
print(f"Extraction took {elapsed:.1f}s")

### Inspect the VLM's JSON output

Parses the raw VLM response (strips markdown fences if present) and pretty-prints the structured JSON. Check that tables, narratives, stakeholders etc. are being extracted correctly before running the batch.


In [ ]:
import json

# Strip markdown code fences if present
cleaned = raw_result.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.split("\n", 1)[1]
    cleaned = cleaned.rsplit("```", 1)[0]

try:
    parsed = json.loads(cleaned)
    print("Valid JSON from VLM\n")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e}\n")
    print("Raw output:")
    print(raw_result)

### Test filename metadata parser

Verifies that the structured filename is parsed correctly into Drawer, AwardID, Field2-5, DocumentType, and DocumentTypeShort. Compare against expected values.


In [ ]:
# Quick test: parse the current document filename
meta = parse_filename(DOC_PATH.name)
print("FileNameMetaData:")
for k, v in meta.items():
    print(f"  {k}: {v!r}")

## 8. Batch process all PDFs in /ocr/

Processes every PDF in the `/ocr/` folder and saves JSONL outputs.

### Configure paths

Set the directories for input PDFs, VLM output, and Gemini reference JSONs. The batch processor will find all `.pdf` files in `PDF_DIR` and save `.jsonl` outputs to `VLM_OUT_DIR`. The comparison step matches files by PDF filename stem.


In [ ]:
import json, fitz
from PIL import Image



def load_json_or_jsonl(path):
    """Load a JSON or JSONL file. Returns a dict (first object if JSONL/array)."""
    text = path.read_text().strip()
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data[0] if data else {}
        return data
    except json.JSONDecodeError:
        pass
    first_line = text.split("\n")[0].strip()
    data = json.loads(first_line)
    if isinstance(data, list):
        return data[0] if data else {}
    return data

# ── Configurable paths ────────────────────────────────────────────────
PDF_DIR = Path("/ocr")                          # PDFs to process
VLM_OUT_DIR = Path("/ocr/vlm_output")           # VLM JSONL output
GEMINI_DIR = Path("/ocr/gemini")                # Gemini reference JSONs

VLM_OUT_DIR.mkdir(exist_ok=True)

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s) in {PDF_DIR}")
for p in pdf_files:
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

if GEMINI_DIR.exists():
    gemini_files = sorted(GEMINI_DIR.glob("*.json")) + sorted(GEMINI_DIR.glob("*.jsonl"))
    print(f"\nFound {len(gemini_files)} Gemini JSON(s) in {GEMINI_DIR}")
    for g in gemini_files:
        print(f"  {g.name}")
else:
    print(f"\nGemini dir not found: {GEMINI_DIR}")
    print("Create it and upload Gemini JSONs to enable comparison.")

### Run batch extraction

Loops through every PDF in the input directory. For each document:
1. Renders all pages at 1x resolution
2. Sends each page image to the VLM with the extraction prompt
3. Parses the JSON response (handles markdown fences and parse failures)
4. Assembles page results into a document-level JSONL and saves to disk

Progress is printed per-page with timing.


In [ ]:
import json, fitz, time
from PIL import Image

SKIP_EXISTING = True  # Set False to reprocess everything

batch_results = []
skipped = 0

for pdf_path in pdf_files:
    out_path = VLM_OUT_DIR / f"{pdf_path.stem}_extracted.jsonl"

    # Skip if already processed
    if SKIP_EXISTING and out_path.exists():
        skipped += 1
        batch_results.append({"filename": pdf_path.name, "output": load_json_or_jsonl(out_path), "output_path": str(out_path)})
        print(f"SKIP (already exists): {pdf_path.name}")
        continue

    print(f"\n{'='*60}")
    print(f"Processing: {pdf_path.name}")
    print(f"{'='*60}")

    doc = fitz.open(str(pdf_path))
    page_count = len(doc)

    # Render all pages and extract links
    images = []
    links_per_page = []
    for i, page in enumerate(doc):
        mat = fitz.Matrix(1.0, 1.0)
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
        links_per_page.append(extract_links(page))
    doc.close()
    print(f"  {page_count} page(s) rendered")

    # Extract each page
    page_results = []
    for i, img in enumerate(images):
        page_num = i + 1
        t0 = time.time()
        raw = extract_page(img, EXTRACTION_PROMPT, filename=pdf_path.name, links=links_per_page[i])
        elapsed = time.time() - t0

        cleaned = raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1]
            cleaned = cleaned.rsplit("```", 1)[0]
        try:
            extracted = json.loads(cleaned)
        except json.JSONDecodeError:
            print(f"    Page {page_num}: INVALID JSON")
            extracted = {"raw_text": raw, "confidence_percentage": 0,
                         "confidence_narrative": "Failed to parse structured output"}

        page_results.append({"page": page_num, "method": "vlm_image",
                             "elapsed_ms": round(elapsed * 1000, 1), "extracted": extracted})
        print(f"    Page {page_num}/{page_count}: {elapsed:.1f}s")

    # Assemble and save
    output = assemble_document_jsonl(pdf_path.name, page_results, MODEL_NAME, prompt_text=EXTRACTION_PROMPT)
    out_path.write_text(json.dumps(output) + "\n")
    batch_results.append({"filename": pdf_path.name, "output": output,
                          "page_results": page_results, "output_path": str(out_path)})
    print(f"  Saved: {out_path.name}")

print(f"\n{'='*60}")
print(f"Batch complete. {len(batch_results)} doc(s) total, {skipped} skipped, {len(batch_results) - skipped} new.")
print(f"Output dir: {VLM_OUT_DIR}")


### Pass 2: Document-level synthesis (optional)

Pass 1 (above) processes each page independently. Pass 2 reads the
per-page JSONs and adds **document-level metadata** that no single
page can provide on its own:

- **Document metadata**: title, creator, date, award number — pulled
  from whichever page has it, applied to the whole document
- **Document summary**: a real summary, not concatenated per-page summaries
- **Cross-page link annotations**: which continuation links were detected

**Safety guarantee:** Pass 2 never modifies per-page extraction results.
It only reads them (as text, no images) and writes new top-level fields.
The per-page data is always the source of truth.

Cross-page table/text linking is already done programmatically in the
assembly function using continuation flags — no LLM needed for that.

In [ ]:
# ── Pass 2: Document-level synthesis ─────────────────────────────────
# Reads per-page JSONs (text only, no images) and produces document-level
# metadata. Uses the same VLM/LLM — just a text-in, text-out call.
#
# This is OPTIONAL — pass 1 output is complete without it.

SYNTHESIS_PROMPT = """You are given the per-page extraction results from a multi-page document.
Each page was processed independently by a VLM. Your job is to produce
ONLY a document-level metadata summary. Do NOT rewrite, merge, or modify
the per-page content — it is the source of truth.

Return a JSON object with:
{
  "document_title": "<best title found across all pages>",
  "document_type": "<e.g. Notice of Award, Grant Guidelines, Book, Manuscript, Sheet Music Collection>",
  "creator": "<author, PI, composer, or agency — whoever created this document>",
  "date": "<most relevant date (publication, award, etc.)>",
  "language": "<primary language>",
  "page_count": <number>,
  "document_summary": "<2-3 sentence summary of the entire document>",
  "key_identifiers": {
    "award_number": "<if grant document>",
    "call_number": "<if library material>",
    "isbn_issn": "<if published work>"
  },
  "cross_page_notes": ["<any observations about content spanning pages, e.g. 'Budget table spans pages 3-5', 'Chapter 2 begins mid-page on p.12'>"]
}

RULES:
- Extract metadata from whichever page contains it (usually page 1 or title page)
- For document_summary, synthesize across ALL pages — do not just repeat page 1
- For cross_page_notes, look at the continuation flags in each page's JSON
- Output ONLY valid JSON. No markdown fences."""


def run_pass2_synthesis(page_results, model_name):
    """Run pass 2 synthesis over per-page JSON results (text-only, no images).

    Returns the synthesis dict, or None if it fails.
    """
    # Build a compact text representation of all pages
    pages_text = []
    for pr in page_results:
        d = pr.get("extracted", {})
        pages_text.append(f"--- PAGE {pr['page']} ---")
        pages_text.append(json.dumps(d, indent=1, ensure_ascii=False))
    all_pages = "\n".join(pages_text)

    # Check if it fits in context (rough estimate: 4 chars ≈ 1 token)
    est_tokens = len(all_pages) // 4
    if est_tokens > 100_000:
        print(f"  [pass2] WARNING: {est_tokens} estimated tokens — may exceed context window")
        print(f"  [pass2] Consider processing in chunks for very large documents")

    messages = [
        {"role": "user", "content": [
            {"type": "text", "text": f"{SYNTHESIS_PROMPT}\n\n{all_pages}"}
        ]}
    ]

    try:
        raw = run_vlm(messages, max_tokens=2048)
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(cleaned)
    except Exception as e:
        print(f"  [pass2] Synthesis failed: {e}")
        return None


# ── Run pass 2 on batch results ──────────────────────────────────────
RUN_PASS2 = True  # Set False to skip pass 2

if RUN_PASS2 and batch_results:
    print(f"Running pass 2 synthesis on {len(batch_results)} documents...\n")
    for br in batch_results:
        if "page_results" not in br:
            continue
        t0 = time.time()
        synthesis = run_pass2_synthesis(br["page_results"], MODEL_NAME)
        elapsed = time.time() - t0
        if synthesis:
            br["output"]["DocumentSynthesis"] = synthesis
            print(f"  {br['filename']}: pass 2 OK ({elapsed:.1f}s) — {synthesis.get('document_type', '?')}")
        else:
            print(f"  {br['filename']}: pass 2 FAILED ({elapsed:.1f}s)")

        # Re-save the JSONL with synthesis added
        out_path = br.get("output_path")
        if out_path:
            with open(out_path, "w") as f:
                json.dump(br["output"], f, indent=2, ensure_ascii=False)

    print("\nPass 2 complete.")
elif not RUN_PASS2:
    print("Pass 2 skipped (RUN_PASS2 = False)")
else:
    print("No batch results to process. Run the batch extraction cell first.")

## 9. Compare VLM vs Gemini extractions

Side-by-side comparison of what our local VLM extracted vs what Gemini produced for the same PDFs. This tells us whether the local model is competitive with Gemini for grant document extraction.

**How it works:**
1. Matches files by PDF name (e.g. `foo.pdf` -> VLM's `foo_extracted.jsonl` vs Gemini's `foo.json`)
2. For each matched pair, compares every section of the output schema
3. Produces a per-document text report + 6-panel visual comparison

**What the metrics mean:**

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Narrative text coverage** | Total characters of verbatim text extracted | More = the model captured more of the actual document content. Critical for RAG — missing text means missing search results. |
| **Text similarity** | How much the VLM and Gemini narrative text overlap (0-100%) | High = both models extracted the same content. Low = one model found text the other missed, OR they structured it differently. |
| **Table cells** | Number of individual data cells extracted from tables | More cells = more complete table extraction. Missing cells = lost dollar amounts, dates, etc. |
| **Stakeholders** | People/orgs identified (PIs, contacts, sponsors) | Checks if the model finds all named individuals and their roles. |
| **Addresses** | Physical addresses found in the document | Checks if letterhead, mailing, and signature block addresses are captured. |
| **Confidence** | The model's self-reported extraction confidence | Self-assessed, so take with a grain of salt. Useful for flagging pages the model struggled with. |
| **Wins scorecard** | Which model "won" more categories across all documents | Quick summary: is VLM or Gemini better overall? |


### Comparison functions

Defines the comparison and plotting code. Run this cell to load the functions — the next cell actually executes the comparison.


In [ ]:
import json
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import numpy as np

def compare_extractions(vlm, gem, filename):
    """Compare VLM vs Gemini extraction for a single document. Returns report dict."""
    report = {"filename": filename, "sections": {}}

    # ── Narrative coverage
    vlm_narr = vlm.get("NarrativeResponses", [])
    gem_narr = gem.get("NarrativeResponses", [])
    vlm_text = " ".join(n.get("VerbatimText", "") for n in vlm_narr)
    gem_text = " ".join(n.get("VerbatimText", "") for n in gem_narr)
    text_sim = SequenceMatcher(None, vlm_text.lower(), gem_text.lower()).ratio() if (vlm_text and gem_text) else 0

    report["sections"]["NarrativeResponses"] = {
        "vlm_count": len(vlm_narr), "gemini_count": len(gem_narr),
        "vlm_total_chars": len(vlm_text), "gemini_total_chars": len(gem_text),
        "text_similarity": round(text_sim * 100, 1),
        "vlm_has_citations": "[cite:" in vlm_text,
        "gemini_has_citations": "[cite:" in gem_text,
    }

    # ── Tables
    vlm_tables = vlm.get("TablesCollection", [])
    gem_tables = gem.get("TablesCollection", [])

    def count_table_cells(tables):
        total = 0
        for t in tables:
            td = t.get("TableData", [])
            if isinstance(td, list):
                for row in td:
                    total += len(row) if isinstance(row, (dict, list)) else 1
            elif isinstance(td, dict):
                total += len(td)
        return total

    report["sections"]["TablesCollection"] = {
        "vlm_count": len(vlm_tables), "gemini_count": len(gem_tables),
        "vlm_total_cells": count_table_cells(vlm_tables),
        "gemini_total_cells": count_table_cells(gem_tables),
        "vlm_classifications": [t.get("TableClassification", "?") for t in vlm_tables],
        "gemini_classifications": [t.get("TableClassification", "?") for t in gem_tables],
    }

    # ── Stakeholders
    vlm_stak = vlm.get("Stakeholders", [])
    gem_stak = gem.get("Stakeholders", [])
    vlm_names = sorted(set(s.get("FullName", "") for s in vlm_stak if s.get("FullName")))
    gem_names = sorted(set(s.get("FullName", "") for s in gem_stak if s.get("FullName")))

    report["sections"]["Stakeholders"] = {
        "vlm_count": len(vlm_stak), "gemini_count": len(gem_stak),
        "vlm_names": vlm_names, "gemini_names": gem_names,
        "names_in_common": sorted(set(vlm_names) & set(gem_names)),
        "vlm_only": sorted(set(vlm_names) - set(gem_names)),
        "gemini_only": sorted(set(gem_names) - set(vlm_names)),
    }

    # ── Addresses
    vlm_addr = vlm.get("AddressesCollection", [])
    gem_addr = gem.get("AddressesCollection", [])
    report["sections"]["AddressesCollection"] = {
        "vlm_count": len(vlm_addr), "gemini_count": len(gem_addr),
    }

    # ── Document tags
    vlm_tags = set(vlm.get("DocumentTags", []))
    gem_tags = set(gem.get("DocumentTags", []))
    report["sections"]["DocumentTags"] = {
        "vlm_tags": sorted(vlm_tags), "gemini_tags": sorted(gem_tags),
        "in_common": sorted(vlm_tags & gem_tags),
        "vlm_only": sorted(vlm_tags - gem_tags),
        "gemini_only": sorted(gem_tags - vlm_tags),
    }

    # ── Document details
    vlm_dd = vlm.get("DocumentDetails", {})
    gem_dd = gem.get("DocumentDetails", {})
    dd_diffs = {}
    for key in set(list(vlm_dd.keys()) + list(gem_dd.keys())):
        v, g = vlm_dd.get(key, ""), gem_dd.get(key, "")
        if str(v).strip() != str(g).strip():
            dd_diffs[key] = {"vlm": v, "gemini": g}

    report["sections"]["DocumentDetails"] = {
        "matching_fields": len(vlm_dd) - len(dd_diffs),
        "differing_fields": dd_diffs,
    }

    # ── Flags
    report["sections"]["Flags"] = {
        "HasAnnotation": {"vlm": vlm.get("HasAnnotation"), "gemini": gem.get("HasAnnotation")},
        "HasWatermark": {"vlm": vlm.get("HasWatermark"), "gemini": gem.get("HasWatermark")},
        "SignatureLines": {"vlm": vlm.get("SignatureLines"), "gemini": gem.get("SignatureLines")},
    }

    # ── Confidence
    report["sections"]["Confidence"] = {
        "vlm": vlm.get("ConfidencePercentage"),
        "gemini": gem.get("ConfidencePercentage"),
        "vlm_narrative": vlm.get("ConfidenceNarrative", "")[:200],
        "gemini_narrative": gem.get("ConfidenceNarrative", "")[:200],
    }

    # ── FileNameMetaData
    vlm_fm = vlm.get("FileNameMetaData", {})
    gem_fm = gem.get("FileNameMetaData", {})
    fm_match = sum(1 for k in vlm_fm if str(vlm_fm.get(k, "")) == str(gem_fm.get(k, "")))

    report["sections"]["FileNameMetaData"] = {
        "matching": fm_match, "total": max(len(vlm_fm), len(gem_fm)),
        "diffs": {k: {"vlm": vlm_fm.get(k, ""), "gemini": gem_fm.get(k, "")}
                  for k in set(list(vlm_fm.keys()) + list(gem_fm.keys()))
                  if str(vlm_fm.get(k, "")) != str(gem_fm.get(k, ""))},
    }

    return report


def print_report(report):
    """Pretty-print a comparison report."""
    print(f"\n{'='*70}")
    print(f"  {report['filename']}")
    print(f"{'='*70}")
    s = report["sections"]

    n = s["NarrativeResponses"]
    print(f"\n  NARRATIVES")
    print(f"    Sections:   VLM={n['vlm_count']}, Gemini={n['gemini_count']}")
    print(f"    Text chars: VLM={n['vlm_total_chars']:,}, Gemini={n['gemini_total_chars']:,}")
    print(f"    Similarity: {n['text_similarity']}%")
    print(f"    Citations:  VLM={'yes' if n['vlm_has_citations'] else 'NO'}, "
          f"Gemini={'yes' if n['gemini_has_citations'] else 'NO'}")
    winner = "VLM" if n["vlm_total_chars"] >= n["gemini_total_chars"] else "Gemini"
    print(f"    >> More text coverage: {winner}")

    t = s["TablesCollection"]
    print(f"\n  TABLES")
    print(f"    Count: VLM={t['vlm_count']}, Gemini={t['gemini_count']}")
    print(f"    Cells: VLM={t['vlm_total_cells']}, Gemini={t['gemini_total_cells']}")

    sk = s["Stakeholders"]
    print(f"\n  STAKEHOLDERS")
    print(f"    Count: VLM={sk['vlm_count']}, Gemini={sk['gemini_count']}")
    if sk["names_in_common"]: print(f"    Common:      {sk['names_in_common']}")
    if sk["vlm_only"]: print(f"    VLM only:    {sk['vlm_only']}")
    if sk["gemini_only"]: print(f"    Gemini only: {sk['gemini_only']}")

    a = s["AddressesCollection"]
    print(f"\n  ADDRESSES: VLM={a['vlm_count']}, Gemini={a['gemini_count']}")

    dd = s["DocumentDetails"]
    print(f"\n  DOCUMENT DETAILS: {dd['matching_fields']} matching")
    for k, v in dd["differing_fields"].items():
        print(f"    {k}: VLM={v['vlm']!r} vs Gemini={v['gemini']!r}")

    c = s["Confidence"]
    print(f"\n  CONFIDENCE: VLM={c['vlm']}%, Gemini={c['gemini']}%")

    fm = s["FileNameMetaData"]
    print(f"  FILENAME METADATA: {fm['matching']}/{fm['total']} matching")


def plot_comparison(reports, model_name):
    """Generate comparison plots for VLM vs Gemini."""
    if not reports:
        print("No reports to plot.")
        return

    doc_names = [r["filename"][:40] for r in reports]
    n_docs = len(reports)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"VLM ({model_name.split('/')[-1]}) vs Gemini", fontsize=14, fontweight="bold")

    colors_vlm = "#2196F3"
    colors_gem = "#FF9800"

    # 1. Narrative text coverage (chars)
    ax = axes[0, 0]
    vlm_chars = [r["sections"]["NarrativeResponses"]["vlm_total_chars"] for r in reports]
    gem_chars = [r["sections"]["NarrativeResponses"]["gemini_total_chars"] for r in reports]
    x = np.arange(n_docs)
    w = 0.35
    ax.barh(x - w/2, vlm_chars, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_chars, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Characters")
    ax.set_title("Narrative Text Coverage")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 2. Text similarity
    ax = axes[0, 1]
    sims = [r["sections"]["NarrativeResponses"]["text_similarity"] for r in reports]
    bars = ax.barh(x, sims, color=["#4CAF50" if s >= 70 else "#FFC107" if s >= 40 else "#F44336" for s in sims])
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Similarity %")
    ax.set_title("Narrative Text Similarity")
    ax.set_xlim(0, 100)
    ax.axvline(70, color="gray", linestyle="--", alpha=0.5)
    ax.invert_yaxis()

    # 3. Table cells
    ax = axes[0, 2]
    vlm_cells = [r["sections"]["TablesCollection"]["vlm_total_cells"] for r in reports]
    gem_cells = [r["sections"]["TablesCollection"]["gemini_total_cells"] for r in reports]
    ax.barh(x - w/2, vlm_cells, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_cells, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Cells extracted")
    ax.set_title("Table Extraction (cells)")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 4. Stakeholders + Addresses
    ax = axes[1, 0]
    vlm_sk = [r["sections"]["Stakeholders"]["vlm_count"] for r in reports]
    gem_sk = [r["sections"]["Stakeholders"]["gemini_count"] for r in reports]
    vlm_ad = [r["sections"]["AddressesCollection"]["vlm_count"] for r in reports]
    gem_ad = [r["sections"]["AddressesCollection"]["gemini_count"] for r in reports]
    w2 = 0.2
    ax.barh(x - 1.5*w2, vlm_sk, w2, label="VLM Stakeholders", color=colors_vlm)
    ax.barh(x - 0.5*w2, gem_sk, w2, label="Gemini Stakeholders", color=colors_gem)
    ax.barh(x + 0.5*w2, vlm_ad, w2, label="VLM Addresses", color=colors_vlm, alpha=0.5)
    ax.barh(x + 1.5*w2, gem_ad, w2, label="Gemini Addresses", color=colors_gem, alpha=0.5)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Count")
    ax.set_title("Stakeholders & Addresses")
    ax.legend(fontsize=7)
    ax.invert_yaxis()

    # 5. Confidence scores
    ax = axes[1, 1]
    vlm_conf = [r["sections"]["Confidence"]["vlm"] or 0 for r in reports]
    gem_conf = [r["sections"]["Confidence"]["gemini"] or 0 for r in reports]
    ax.barh(x - w/2, vlm_conf, w, label="VLM", color=colors_vlm)
    ax.barh(x + w/2, gem_conf, w, label="Gemini", color=colors_gem)
    ax.set_yticks(x)
    ax.set_yticklabels(doc_names, fontsize=7)
    ax.set_xlabel("Confidence %")
    ax.set_title("Self-Reported Confidence")
    ax.set_xlim(0, 100)
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    # 6. Wins scorecard
    ax = axes[1, 2]
    categories = ["Narratives", "Tables", "Stakeholders", "Addresses"]
    vlm_w = [0, 0, 0, 0]
    gem_w = [0, 0, 0, 0]
    for r in reports:
        s = r["sections"]
        n = s["NarrativeResponses"]
        if n["vlm_total_chars"] > n["gemini_total_chars"]: vlm_w[0] += 1
        elif n["gemini_total_chars"] > n["vlm_total_chars"]: gem_w[0] += 1
        t = s["TablesCollection"]
        if t["vlm_total_cells"] > t["gemini_total_cells"]: vlm_w[1] += 1
        elif t["gemini_total_cells"] > t["vlm_total_cells"]: gem_w[1] += 1
        sk = s["Stakeholders"]
        if sk["vlm_count"] > sk["gemini_count"]: vlm_w[2] += 1
        elif sk["gemini_count"] > sk["vlm_count"]: gem_w[2] += 1
        a = s["AddressesCollection"]
        if a["vlm_count"] > a["gemini_count"]: vlm_w[3] += 1
        elif a["gemini_count"] > a["vlm_count"]: gem_w[3] += 1
    y = np.arange(len(categories))
    ax.barh(y - w/2, vlm_w, w, label="VLM wins", color=colors_vlm)
    ax.barh(y + w/2, gem_w, w, label="Gemini wins", color=colors_gem)
    ax.set_yticks(y)
    ax.set_yticklabels(categories)
    ax.set_xlabel("Documents won")
    ax.set_title("Wins Scorecard")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

print("Comparison functions ready (with plots).")


### Run comparison and generate plots

Runs the comparison for all matched VLM/Gemini pairs and prints:
- **Per-document report**: detailed breakdown of each section (narratives, tables, stakeholders, etc.)
- **Aggregate scorecard**: tallies which model wins each category across all documents
- **6-panel chart**: visual comparison — narrative coverage, text similarity, table completeness, stakeholders/addresses, confidence, and overall wins


In [ ]:
# ── Run comparison ────────────────────────────────────────────────────
reports = []
matched = 0
unmatched_vlm = []
unmatched_gem = []

# Build lookup: PDF stem -> VLM output
vlm_files = {p.stem.replace("_extracted", ""): p
             for p in VLM_OUT_DIR.glob("*.jsonl")}

# Build lookup: PDF stem -> Gemini output (try multiple naming conventions)
gem_lookup = {}
if GEMINI_DIR.exists():
    for g in list(GEMINI_DIR.glob("*.json")) + list(GEMINI_DIR.glob("*.jsonl")):
        stem = g.stem.replace("_extracted", "").replace("_gemini", "")
        gem_lookup[stem] = g

# Match and compare
for stem, vlm_path in sorted(vlm_files.items()):
    if stem in gem_lookup:
        matched += 1
        vlm_data = load_json_or_jsonl(vlm_path)
        gem_data = load_json_or_jsonl(gem_lookup[stem])
        report = compare_extractions(vlm_data, gem_data, vlm_data.get("Filename", stem))
        reports.append(report)
        print_report(report)
    else:
        unmatched_vlm.append(stem)

for stem in gem_lookup:
    if stem not in vlm_files:
        unmatched_gem.append(stem)

# ── Aggregate summary ─────────────────────────────────────────────────
print(f"\n\n{'#'*70}")
print(f"  AGGREGATE SUMMARY — {len(reports)} document(s) compared")
print(f"{'#'*70}")

if unmatched_vlm:
    print(f"\n  VLM outputs with no Gemini match: {unmatched_vlm}")
if unmatched_gem:
    print(f"  Gemini JSONs with no VLM match: {unmatched_gem}")

if reports:
    # Tally wins per section
    vlm_wins = {"narratives": 0, "tables": 0, "stakeholders": 0, "addresses": 0}
    gem_wins = {"narratives": 0, "tables": 0, "stakeholders": 0, "addresses": 0}
    total_sim = 0

    for r in reports:
        s = r["sections"]
        n = s["NarrativeResponses"]
        total_sim += n["text_similarity"]
        if n["vlm_total_chars"] > n["gemini_total_chars"]:
            vlm_wins["narratives"] += 1
        elif n["gemini_total_chars"] > n["vlm_total_chars"]:
            gem_wins["narratives"] += 1

        t = s["TablesCollection"]
        if t["vlm_total_cells"] > t["gemini_total_cells"]:
            vlm_wins["tables"] += 1
        elif t["gemini_total_cells"] > t["vlm_total_cells"]:
            gem_wins["tables"] += 1

        sk = s["Stakeholders"]
        if sk["vlm_count"] > sk["gemini_count"]:
            vlm_wins["stakeholders"] += 1
        elif sk["gemini_count"] > sk["vlm_count"]:
            gem_wins["stakeholders"] += 1

        a = s["AddressesCollection"]
        if a["vlm_count"] > a["gemini_count"]:
            vlm_wins["addresses"] += 1
        elif a["gemini_count"] > a["vlm_count"]:
            gem_wins["addresses"] += 1

    n_docs = len(reports)
    print(f"\n  Average narrative text similarity: {total_sim / n_docs:.1f}%")
    print(f"\n  {'Category':<20} {'VLM wins':<12} {'Gemini wins':<12} {'Tie':<8}")
    print(f"  {'-'*52}")
    for cat in vlm_wins:
        ties = n_docs - vlm_wins[cat] - gem_wins[cat]
        print(f"  {cat:<20} {vlm_wins[cat]:<12} {gem_wins[cat]:<12} {ties:<8}")

    total_vlm = sum(vlm_wins.values())
    total_gem = sum(gem_wins.values())
    print(f"  {'-'*52}")
    print(f"  {'TOTAL':<20} {total_vlm:<12} {total_gem:<12} "
          f"{n_docs * 4 - total_vlm - total_gem:<8}")

    if total_vlm > total_gem:
        print(f"\n  >> Overall winner: VLM ({MODEL_NAME})")
    elif total_gem > total_vlm:
        print(f"\n  >> Overall winner: Gemini")
    else:
        print(f"\n  >> Result: TIE")
else:
    print("\n  No matched documents to compare.")
    print("  Upload Gemini JSONs to /ocr/gemini/ with filenames matching the PDFs.")

# ── Plots ─────────────────────────────────────────────────────────────
plot_comparison(reports, MODEL_NAME)


## 10. Launch Streamlit app

Tests the extraction server + Streamlit UI stack from within this notebook — same code that runs in production as separate jobs.

**Prerequisites:**
1. Your workspace must have a **Custom URL tool on port 8501** configured (in addition to Jupyter on 8888). Ask your admin to add this if you don't have it.
2. Set env var `STREAMLIT_BASE_PATH` to the tool's URL path. Check the RunAI UI — click the Custom URL tool link to see the actual path (usually `/<project>/<job-name>/url-1`).
3. The model will be freed from GPU and reloaded by the server process.

**To add the tool:** RunAI UI > your workspace > Edit > Tools > Add Tool > Custom URL, port 8501

**To find the URL path:** RunAI UI > your workspace > Connect > click the Custom URL tool link. The path in the URL after the cluster host is your `STREAMLIT_BASE_PATH`.


In [ ]:
import subprocess

repo_dir = "/ocr/repo"

# Free notebook model from GPU before server loads its own
if 'model' in dir() and model is not None:
    del model
    del processor
    torch.cuda.empty_cache()
    print("Freed notebook model from GPU.")

# Start extraction server in local mode (loads model with transformers, no vLLM)
env = {**os.environ, "LLM_BASE_URL": "local", "HF_HUB_OFFLINE": "1"}
server_proc = subprocess.Popen(
    ["python", "ocr_app/scripts/ocr_server.py"],
    env=env, cwd=repo_dir,
)
print(f"Extraction server starting (PID {server_proc.pid})...")
print("  Mode: LOCAL (transformers, no vLLM)")
print(f"  Model: {MODEL_NAME}")
print("  All pages -> VLM image extraction")

# Wait for server to load model and be ready
import time as _time
for i in range(120):
    _time.sleep(2)
    try:
        import httpx
        resp = httpx.get("http://localhost:8090/health", timeout=2.0)
        if resp.status_code == 200:
            info = resp.json()
            print(f"\nServer ready! LLM: {info.get('llm_model', '?')}")
            break
    except Exception:
        if i % 15 == 14:
            print(f"  Still loading model... ({(i+1)*2}s)")
else:
    print("WARNING: Server did not become ready within 4 min")

# Start Streamlit UI with proxy-compatible base path
base_path = os.environ.get("STREAMLIT_BASE_PATH", "")
streamlit_cmd = [
    "streamlit", "run", "ocr_app/app.py",
    "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true",
]
if base_path:
    streamlit_cmd.append(f"--server.baseUrlPath={base_path}")

streamlit_proc = subprocess.Popen(
    streamlit_cmd,
    env={**os.environ, "OCR_SERVICE_URL": "http://localhost:8090"},
    cwd=repo_dir,
)
print(f"Streamlit started (PID {streamlit_proc.pid})")
if base_path:
    print(f"\nAccess: https://deepthought.doit.wisc.edu{base_path}/")
else:
    print("\nWARNING: STREAMLIT_BASE_PATH not set — proxy URL will 404")
    print("Add env var: STREAMLIT_BASE_PATH = /<project>/<job-name>/url-1")
print("\nTo stop: server_proc.terminate(); streamlit_proc.terminate()")
